# LeetCode #92 — Reverse Linked List II: đảo ngược một đoạn trong một lượt

**Mức độ:** Medium · **Chủ đề:** Linked List

Skeleton trên LeetCode:

```python
class Solution:
    def reverseBetween(self, head: Optional[ListNode], left: int, right: int) -> Optional[ListNode]:
```

Mục tiêu của notebook:

1. Ôn lại đảo ngược **toàn bộ** danh sách (#206) với `prev`, `cur`, `nxt`.
2. Đảo một **đoạn** `[left, right]` rồi **nối lại** hai đầu.
3. Kỹ thuật **chèn lên đầu (head insertion)**: một lượt, không cần tìm đoạn trước.
4. Dùng **nút giả** để xử lý `left = 1`.

# 1. Bài toán là gì?

Cho `head` của danh sách liên kết đơn và hai số `left <= right`. Đảo ngược các nút từ **vị trí** `left` tới **vị trí** `right` (đánh số từ 1). Trả về danh sách sau khi đảo.

**Constraints (theo đề):**

- Số nút là `n`, `1 <= n <= 500`
- `-500 <= Node.val <= 500`
- `1 <= left <= right <= n`

**Follow up (theo đề):** làm được trong **một lượt** không?

---

## Ví dụ

```text
1 -> 2 -> 3 -> 4 -> 5,  left = 2, right = 4   ->   1 -> 4 -> 3 -> 2 -> 5
5,                      left = 1, right = 1   ->   5
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Danh sách liên kết đơn (singly linked list)

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Danh sách liên kết đơn (singly linked list)

### Danh sách liên kết là gì?

Mảng (`list` của Python) lưu các phần tử **liền nhau** trong bộ nhớ, nên truy cập `a[i]` tức thì. Danh sách liên kết thì khác: mỗi phần tử là một **nút** (node) riêng lẻ, chứa **giá trị** và **con trỏ** `next` chỉ tới nút kế tiếp. Nút cuối có `next = None`.

```text
head
 │
 ▼
┌───┬───┐   ┌───┬───┐   ┌───┬────┐
│ 1 │ ●─┼──►│ 2 │ ●─┼──►│ 3 │None│
└───┴───┘   └───┴───┘   └───┴────┘
 val next
```

Ta chỉ giữ con trỏ tới nút đầu (`head`). Muốn tới nút thứ `k` phải đi qua `k` bước — **không** có truy cập ngẫu nhiên.

| | Mảng | Danh sách liên kết |
|---|---|---|
| Lấy phần tử thứ `i` | `O(1)` | `O(i)` |
| Chèn/xóa ở đầu | `O(n)` | `O(1)` |
| Chèn/xóa sau một nút đã có trong tay | `O(n)` | `O(1)` |

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

# Tạo bằng tay: 1 -> 2 -> 3
c = ListNode(3)
b = ListNode(2, c)
a = ListNode(1, b)
head = a
print(head.val, head.next.val, head.next.next.val, head.next.next.next)

1 2 3 None


### Duyệt danh sách và hai hàm tiện ích

Mẫu duyệt cơ bản: con trỏ `cur` bắt đầu ở `head`, mỗi bước `cur = cur.next`, dừng khi `cur is None`.

In [2]:
def tao_danh_sach(values):
    """[1, 2, 3] -> 1 -> 2 -> 3, trả về head."""
    dummy = tail = ListNode()          # nút giả để khỏi xử lý riêng nút đầu
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next

def ra_list(head):
    out, cur = [], head
    while cur:                          # duyệt tới khi hết
        out.append(cur.val)
        cur = cur.next
    return out

h = tao_danh_sach([4, 5, 6, 7])
print("ra_list:", ra_list(h))
dem, cur = 0, h
while cur:
    dem += 1
    cur = cur.next
print("độ dài:", dem)

ra_list: [4, 5, 6, 7]
độ dài: 4


### Chèn và xóa: chỉ cần đổi con trỏ

Chèn nút `x` **sau** nút `p`: `x.next = p.next; p.next = x` (thứ tự quan trọng — làm ngược sẽ mất phần sau).

Xóa nút **sau** `p`: `p.next = p.next.next`.

```text
Chèn x sau p:   p ──► q        ⟹   p ──► x ──► q
Xóa sau p:      p ──► q ──► r  ⟹   p ──────────► r
```

Muốn xóa một nút, ta cần nút **đứng trước** nó. Nếu nút cần xóa là `head` thì không có nút trước → dùng **nút giả (dummy)** đặt trước `head`.

In [3]:
h = tao_danh_sach([1, 2, 4])
p = h.next                              # nút 2
x = ListNode(3)
x.next = p.next                         # 3 -> 4
p.next = x                              # 2 -> 3
print("sau khi chèn 3:", ra_list(h))

def xoa_gia_tri(head, target):
    dummy = ListNode(0, head)           # dummy đứng trước head
    prev = dummy
    while prev.next:
        if prev.next.val == target:
            prev.next = prev.next.next  # bỏ qua nút cần xóa
        else:
            prev = prev.next
    return dummy.next                   # head mới (có thể đã đổi)

print("xóa mọi số 1:", ra_list(xoa_gia_tri(tao_danh_sach([1, 1, 2, 1, 3]), 1)))

sau khi chèn 3: [1, 2, 3, 4]
xóa mọi số 1: [2, 3]


### Đảo ngược danh sách — kỹ thuật ba con trỏ

Đi dọc danh sách, lật mũi tên của từng nút cho trỏ về nút **trước** nó. Cần lưu `nxt` trước khi lật, vì sau khi lật ta mất đường đi tiếp.

```text
prev  cur   nxt
None   1 ──► 2 ──► 3          bước 1: 1.next = None
None ◄─1     2 ──► 3          bước 2: 2.next = 1
None ◄─1 ◄─ 2     3          bước 3: 3.next = 2  → head mới = 3
```

In [4]:
def dao_nguoc(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next      # 1) lưu đường đi tiếp
        cur.next = prev     # 2) lật mũi tên
        prev = cur          # 3) tiến prev
        cur = nxt           # 4) tiến cur
    return prev             # prev là head mới

print("đảo [1..5]:", ra_list(dao_nguoc(tao_danh_sach([1, 2, 3, 4, 5]))))

đảo [1..5]: [5, 4, 3, 2, 1]


### Con trỏ nhanh / chậm (fast & slow)

`slow` đi 1 bước, `fast` đi 2 bước mỗi vòng.

- Khi `fast` tới cuối, `slow` ở **giữa** danh sách.
- Nếu danh sách có **chu trình**, `fast` sẽ đuổi kịp `slow` (thuật toán Floyd).

In [5]:
def nut_giua(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

print("giữa [1..5]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5])).val,
      "| giữa [1..6]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5, 6])).val, "(nút giữa thứ hai)")

def co_chu_trinh(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False

v = tao_danh_sach([1, 2, 3, 4])
v.next.next.next.next = v.next          # 4 -> 2 tạo chu trình
print("có chu trình?", co_chu_trinh(v), "| danh sách thường:", co_chu_trinh(tao_danh_sach([1, 2])))

giữa [1..5]: 3 | giữa [1..6]: 4 (nút giữa thứ hai)
có chu trình? True | danh sách thường: False


### Lỗi hay gặp với danh sách liên kết

- Truy cập `cur.next.val` khi `cur.next` là `None` → `AttributeError`. Luôn kiểm tra `cur and cur.next`.
- Đổi `cur.next` **trước** khi lưu nút kế → mất phần còn lại của danh sách.
- Quên trường hợp danh sách rỗng (`head is None`) hoặc chỉ một nút.
- Quên dùng dummy khi `head` có thể bị xóa/thay → phải viết nhánh riêng, dễ sai.
- In danh sách có chu trình bằng vòng `while cur` → lặp vô hạn.

## 2.2 Áp dụng vào bài này

Đảo một **đoạn** `[left, right]` của **danh sách liên kết**: dùng **nút giả** (vì đoạn có thể bắt đầu từ `head`), đi tới nút **đứng trước** đoạn, rồi đảo đoạn — hoặc bằng ba con trỏ như đảo cả danh sách rồi nối hai đầu lại, hoặc bằng cách lần lượt **chèn** từng nút lên đầu đoạn trong một lượt.

# 3. Nền tảng

In [6]:
from typing import Optional, List
import random


class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def build_list(values: List[int]) -> Optional[ListNode]:
    dummy = ListNode()
    tail = dummy
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next


def to_list(head: Optional[ListNode], limit: int = 1000) -> List[int]:
    out = []
    while head and len(out) < limit:
        out.append(head.val)
        head = head.next
    return out

## 3.1 Đảo ngược toàn bộ (#206)

Đi từng nút, **quay ngược** con trỏ `next` của nó về nút trước:

```text
prev = None, cur = head
while cur:
    nxt = cur.next      # nhớ nút sau trước khi cắt
    cur.next = prev     # quay ngược
    prev = cur          # tiến prev
    cur = nxt           # tiến cur
return prev
```

In [7]:
def reverse_all(head: Optional[ListNode]) -> Optional[ListNode]:
    prev, cur = None, head
    while cur:
        nxt = cur.next
        cur.next = prev
        prev = cur
        cur = nxt
    return prev

print(to_list(reverse_all(build_list([1, 2, 3, 4, 5]))))

[5, 4, 3, 2, 1]


# 4. Baseline: đổi qua list Python

Đúng nhưng tạo nút mới và tốn `O(n)` bộ nhớ.

In [8]:
def reverse_between_list(head: Optional[ListNode], left: int, right: int) -> Optional[ListNode]:
    v = to_list(head)
    v[left - 1:right] = v[left - 1:right][::-1]
    return build_list(v)

print(to_list(reverse_between_list(build_list([1, 2, 3, 4, 5]), 2, 4)))

[1, 4, 3, 2, 5]


# 5. Cách 1: Cắt đoạn, đảo, nối lại

1. Dùng nút giả `dummy`, đi `left − 1` bước tới `before` (nút đứng **trước** đoạn).
2. Đảo `right − left + 1` nút bắt đầu từ `start = before.next` (giống #206, nhưng dừng sau đúng số nút).
3. Sau khi đảo: `prev` là **đầu mới** của đoạn, `cur` là nút **sau** đoạn. `start` giờ là **đuôi** của đoạn.
4. Nối: `before.next = prev`, `start.next = cur`.

```text
dummy -> 1 -> [2 -> 3 -> 4] -> 5
        before  start            cur (sau khi đảo)
dummy -> 1 -> [4 -> 3 -> 2] -> 5
               prev      start
```

In [9]:
def reverse_between_cut(head: Optional[ListNode], left: int, right: int) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    before = dummy
    for _ in range(left - 1):
        before = before.next
    start = before.next
    prev, cur = None, start
    for _ in range(right - left + 1):
        nxt = cur.next
        cur.next = prev
        prev = cur
        cur = nxt
    before.next = prev
    start.next = cur
    return dummy.next

print(to_list(reverse_between_cut(build_list([1, 2, 3, 4, 5]), 2, 4)))
print(to_list(reverse_between_cut(build_list([1, 2, 3, 4, 5]), 1, 5)))

[1, 4, 3, 2, 5]
[5, 4, 3, 2, 1]


# 6. Cách 2: Chèn lên đầu đoạn (head insertion)

Giữ `before` cố định và `start` (nút đầu đoạn ban đầu — cuối cùng nó sẽ thành đuôi). Lặp `right − left` lần: **lấy nút ngay sau `start`** và **chèn nó vào ngay sau `before`**.

```text
1 -> [2] -> 3 -> 4 -> 5        lấy 3, chèn sau 1
1 -> [3 -> 2] -> 4 -> 5        lấy 4, chèn sau 1
1 -> [4 -> 3 -> 2] -> 5        xong
```

Mỗi lần chèn chỉ đổi 3 con trỏ:

```text
move = start.next
start.next = move.next     # bỏ move ra khỏi vị trí cũ
move.next = before.next    # move trỏ tới đầu đoạn hiện tại
before.next = move         # move thành đầu đoạn
```

In [10]:
def reverse_between_insert(head: Optional[ListNode], left: int, right: int) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    before = dummy
    for _ in range(left - 1):
        before = before.next
    start = before.next
    for _ in range(right - left):
        move = start.next
        start.next = move.next
        move.next = before.next
        before.next = move
    return dummy.next

print(to_list(reverse_between_insert(build_list([1, 2, 3, 4, 5]), 2, 4)))

[1, 4, 3, 2, 5]


# 7. Mô phỏng

In [11]:
def trace_insert(values: List[int], left: int, right: int) -> List[int]:
    dummy = ListNode("D", build_list(values))
    before = dummy
    for _ in range(left - 1):
        before = before.next
    start = before.next
    print(f"before = {before.val}, start = {start.val}")
    print("  ban đầu:", to_list(dummy.next))
    for step in range(right - left):
        move = start.next
        start.next = move.next
        move.next = before.next
        before.next = move
        print(f"  bước {step + 1}: chèn {move.val} sau {before.val} ->", to_list(dummy.next))
    return to_list(dummy.next)

_ = trace_insert([1, 2, 3, 4, 5, 6], 2, 5)

before = 1, start = 2
  ban đầu: [1, 2, 3, 4, 5, 6]
  bước 1: chèn 3 sau 1 -> [1, 3, 2, 4, 5, 6]
  bước 2: chèn 4 sau 1 -> [1, 4, 3, 2, 5, 6]
  bước 3: chèn 5 sau 1 -> [1, 5, 4, 3, 2, 6]


`start` (= nút `2`) không bao giờ di chuyển — nó từ từ bị "đẩy" về cuối đoạn khi các nút sau nó lần lượt được chèn lên đầu.

# 8. Phiên bản đúng format LeetCode

In [12]:
class Solution:
    def reverseBetween(self, head: Optional[ListNode], left: int, right: int) -> Optional[ListNode]:
        dummy = ListNode(0, head)
        before = dummy
        for _ in range(left - 1):
            before = before.next
        start = before.next
        for _ in range(right - left):
            move = start.next
            start.next = move.next
            move.next = before.next
            before.next = move
        return dummy.next

In [13]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS  {actual}")


def run(values, left, right):
    return to_list(Solution().reverseBetween(build_list(values), left, right))


check("Ví dụ 1", run([1, 2, 3, 4, 5], 2, 4), [1, 4, 3, 2, 5])
check("Ví dụ 2", run([5], 1, 1), [5])
check("left = 1", run([1, 2, 3], 1, 2), [2, 1, 3])
check("right = n", run([1, 2, 3], 2, 3), [1, 3, 2])
check("Toàn bộ", run([1, 2, 3, 4], 1, 4), [4, 3, 2, 1])
check("left = right", run([1, 2, 3], 2, 2), [1, 2, 3])

random.seed(92)
for _ in range(5000):
    n = random.randint(1, 10)
    values = [random.randint(-5, 5) for _ in range(n)]
    left = random.randint(1, n)
    right = random.randint(left, n)
    expected = to_list(reverse_between_list(build_list(values), left, right))
    assert run(values, left, right) == expected
    assert to_list(reverse_between_cut(build_list(values), left, right)) == expected
print("5000 test ngẫu nhiên: hai cách khớp cách dùng list")

Ví dụ 1                PASS  [1, 4, 3, 2, 5]
Ví dụ 2                PASS  [5]
left = 1               PASS  [2, 1, 3]
right = n              PASS  [1, 3, 2]
Toàn bộ                PASS  [4, 3, 2, 1]
left = right           PASS  [1, 2, 3]


5000 test ngẫu nhiên: hai cách khớp cách dùng list

# 9. Vì sao đúng? Độ phức tạp?

**Chèn lên đầu — bất biến:** sau `t` lần lặp, đoạn giữa `before` và `start` (bao gồm cả hai đầu của đoạn) là `t + 1` nút đầu tiên của đoạn gốc **theo thứ tự ngược**, và `start.next` là nút tiếp theo chưa xử lý. Sau `right − left` lần: cả đoạn đã đảo, `start` là đuôi và trỏ đúng tới nút sau đoạn.

Cả hai cách đều là **một lượt** (mỗi nút được đi qua tối đa một lần).

```text
Time:  O(n)
Space: O(1)
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Không dùng dummy → sai khi `left = 1`

In [14]:
def wrong_no_dummy(head: Optional[ListNode], left: int, right: int) -> Optional[ListNode]:
    before = head                                   # SAI: coi head là nút trước đoạn
    for _ in range(left - 2):
        before = before.next
    start = before.next
    for _ in range(right - left):
        move = start.next
        start.next = move.next
        move.next = before.next
        before.next = move
    return head

print("Sai :", to_list(wrong_no_dummy(build_list([1, 2, 3, 4, 5]), 1, 3)), "| Đúng:", run([1, 2, 3, 4, 5], 1, 3))

Sai : [1, 4, 3, 2, 5] | Đúng: [3, 2, 1, 4, 5]


## Lỗi 2: Quên nối `start.next = cur` ở cách 1

In [15]:
def wrong_no_reconnect(head: Optional[ListNode], left: int, right: int) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    before = dummy
    for _ in range(left - 1):
        before = before.next
    start = before.next
    prev, cur = None, start
    for _ in range(right - left + 1):
        nxt = cur.next
        cur.next = prev
        prev = cur
        cur = nxt
    before.next = prev                              # SAI: thiếu start.next = cur
    return dummy.next

print("Sai :", to_list(wrong_no_reconnect(build_list([1, 2, 3, 4, 5]), 2, 4)), "| Đúng:", run([1, 2, 3, 4, 5], 2, 4))

Sai : [1, 4, 3, 2] | Đúng: [1, 4, 3, 2, 5]


Nút `5` bị mất: đuôi đoạn (`2`) trỏ tới `None` (giá trị `prev` ban đầu).

## Lỗi 3: Lặp `right − left + 1` lần ở cách chèn lên đầu

Cách chèn chỉ cần `right − left` lần (nút đầu đoạn không cần chèn). Lặp thừa một lần sẽ kéo thêm nút **ngoài đoạn** vào.

In [16]:
def wrong_extra_iteration(head: Optional[ListNode], left: int, right: int) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    before = dummy
    for _ in range(left - 1):
        before = before.next
    start = before.next
    for _ in range(right - left + 1):               # SAI: thừa 1
        move = start.next
        start.next = move.next
        move.next = before.next
        before.next = move
    return dummy.next

print("Sai :", to_list(wrong_extra_iteration(build_list([1, 2, 3, 4, 5]), 2, 4)), "| Đúng:", run([1, 2, 3, 4, 5], 2, 4))

Sai : [1, 5, 4, 3, 2] | Đúng: [1, 4, 3, 2, 5]


## Lỗi 4: Nhầm vị trí (1-based) với chỉ số (0-based)

`left`, `right` đếm từ **1**. Để tới nút **trước** vị trí `left` từ `dummy` (vị trí 0), đi đúng `left − 1` bước.

# 11. Mẹo nhận diện pattern

- **Đảo một đoạn** danh sách liên kết → dummy + tìm `before` + đảo + nối (hoặc chèn lên đầu).
- Ba con trỏ `prev / cur / nxt` là "câu thần chú" cho mọi bài đảo.

Bài liên quan: **#206 Reverse Linked List**, **#24 Swap Nodes in Pairs**, **#25 Reverse Nodes in k-Group** (Hard: lặp lại #92 cho từng nhóm `k`), **#2074 Reverse Nodes in Even Length Groups**, **#234 Palindrome Linked List** (đảo nửa sau).

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Mô phỏng cách chèn lên đầu cho `10 -> 20 -> 30 -> 40`, `left = 1`, `right = 4`.

## Bài 2 — Điền chỗ trống (chèn lên đầu)

In [17]:
def exercise_2_reverse_between(head: Optional[ListNode], left: int, right: int) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    before = dummy
    for _ in range(left - 1):
        before = before.next
    start = before.next
    for _ in range(right - left):
        # TODO: 4 dòng: move, bỏ move ra, move trỏ tới đầu đoạn, move thành đầu đoạn
        pass
    return dummy.next


# assert to_list(exercise_2_reverse_between(build_list([1, 2, 3, 4, 5]), 2, 4)) == [1, 4, 3, 2, 5]
# assert to_list(exercise_2_reverse_between(build_list([5]), 1, 1)) == [5]
# assert to_list(exercise_2_reverse_between(build_list([1, 2, 3]), 1, 3)) == [3, 2, 1]

## Bài 3 — Biến thể: Swap Nodes in Pairs (LeetCode #24)

Đổi chỗ từng cặp nút kề nhau (đổi **nút**, không đổi giá trị).

```python
swap_pairs(1 -> 2 -> 3 -> 4)  -> 2 -> 1 -> 4 -> 3
swap_pairs(1 -> 2 -> 3)       -> 2 -> 1 -> 3
```

In [18]:
def exercise_3_swap_pairs(head: Optional[ListNode]) -> Optional[ListNode]:
    # TODO
    pass


# assert to_list(exercise_3_swap_pairs(build_list([1, 2, 3, 4]))) == [2, 1, 4, 3]
# assert to_list(exercise_3_swap_pairs(build_list([1, 2, 3]))) == [2, 1, 3]
# assert to_list(exercise_3_swap_pairs(build_list([]))) == []

## Bài 4 — Biến thể: Palindrome Linked List (LeetCode #234)

Kiểm tra danh sách có đối xứng không, `O(1)` bộ nhớ: tìm giữa bằng hai con trỏ nhanh/chậm, đảo nửa sau, so sánh.

In [19]:
def exercise_4_is_palindrome(head: Optional[ListNode]) -> bool:
    # TODO
    pass


# assert exercise_4_is_palindrome(build_list([1, 2, 2, 1])) is True
# assert exercise_4_is_palindrome(build_list([1, 2])) is False
# assert exercise_4_is_palindrome(build_list([1, 2, 3, 2, 1])) is True
# assert exercise_4_is_palindrome(build_list([1])) is True

## Bài 5 — Mini challenge: Reverse Nodes in k-Group (LeetCode #25, Hard)

Đảo từng nhóm `k` nút liên tiếp; nhóm cuối nếu ít hơn `k` nút thì giữ nguyên.

```python
reverse_k_group(1 -> 2 -> 3 -> 4 -> 5, 2)  -> 2 -> 1 -> 4 -> 3 -> 5
reverse_k_group(1 -> 2 -> 3 -> 4 -> 5, 3)  -> 3 -> 2 -> 1 -> 4 -> 5
```

Gợi ý nhanh: lặp: kiểm tra còn đủ `k` nút sau `before` không; nếu có, dùng kỹ thuật chèn lên đầu `k − 1` lần; rồi `before = start`.

In [20]:
def exercise_5_reverse_k_group(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    # TODO
    pass


# assert to_list(exercise_5_reverse_k_group(build_list([1, 2, 3, 4, 5]), 2)) == [2, 1, 4, 3, 5]
# assert to_list(exercise_5_reverse_k_group(build_list([1, 2, 3, 4, 5]), 3)) == [3, 2, 1, 4, 5]
# assert to_list(exercise_5_reverse_k_group(build_list([1, 2, 3]), 1)) == [1, 2, 3]

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_insert` sau khi tự làm; cần 3 lần chèn.

**Bài 3**
- Mức 1: đây là #25 với `k = 2`, hoặc #92 lặp lại.
- Mức 2: với `prev = dummy`: `a = prev.next`, `b = a.next`; `prev.next = b; a.next = b.next; b.next = a; prev = a`.

**Bài 4**
- Mức 1: `slow`, `fast`; khi `fast` tới cuối, `slow` ở giữa.
- Mức 2: đảo từ `slow` (hoặc `slow.next`), so từng nút với nửa đầu.

**Bài 5**
- Mức 1: hàm kiểm tra: đi `k` bước từ `before.next`, nếu gặp `None` thì dừng.
- Mức 2: sau khi đảo một nhóm, `start` là đuôi nhóm → `before = start`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [21]:
_ = trace_insert([10, 20, 30, 40], 1, 4)

before = D, start = 10
  ban đầu: [10, 20, 30, 40]
  bước 1: chèn 20 sau D -> [20, 10, 30, 40]
  bước 2: chèn 30 sau D -> [30, 20, 10, 40]
  bước 3: chèn 40 sau D -> [40, 30, 20, 10]


## Đáp án Bài 2

In [22]:
def answer_2_reverse_between(head: Optional[ListNode], left: int, right: int) -> Optional[ListNode]:
    return reverse_between_insert(head, left, right)

assert to_list(answer_2_reverse_between(build_list([1, 2, 3, 4, 5]), 2, 4)) == [1, 4, 3, 2, 5]
assert to_list(answer_2_reverse_between(build_list([5]), 1, 1)) == [5]
assert to_list(answer_2_reverse_between(build_list([1, 2, 3]), 1, 3)) == [3, 2, 1]
print("Bài 2 PASS (giống reverse_between_insert ở mục 6)")

Bài 2 PASS (giống reverse_between_insert ở mục 6)


## Đáp án Bài 3

In [23]:
def answer_3_swap_pairs(head: Optional[ListNode]) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next and prev.next.next:
        a = prev.next
        b = a.next
        prev.next = b
        a.next = b.next
        b.next = a
        prev = a
    return dummy.next

assert to_list(answer_3_swap_pairs(build_list([1, 2, 3, 4]))) == [2, 1, 4, 3]
assert to_list(answer_3_swap_pairs(build_list([1, 2, 3]))) == [2, 1, 3]
assert to_list(answer_3_swap_pairs(build_list([]))) == []
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [24]:
def answer_4_is_palindrome(head: Optional[ListNode]) -> bool:
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    second = reverse_all(slow)
    first = head
    while second:
        if first.val != second.val:
            return False
        first, second = first.next, second.next
    return True

assert answer_4_is_palindrome(build_list([1, 2, 2, 1])) is True
assert answer_4_is_palindrome(build_list([1, 2])) is False
assert answer_4_is_palindrome(build_list([1, 2, 3, 2, 1])) is True
assert answer_4_is_palindrome(build_list([1])) is True
for _ in range(3000):
    v = [random.randint(0, 2) for _ in range(random.randint(1, 8))]
    assert answer_4_is_palindrome(build_list(v)) == (v == v[::-1])
print("Bài 4 PASS")

Bài 4 PASS


(Lời giải này làm thay đổi danh sách — nửa sau bị đảo. Trong thực tế nên đảo lại nửa sau trước khi trả về để không có tác dụng phụ.)

## Đáp án Bài 5

In [25]:
def answer_5_reverse_k_group(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    before = dummy
    while True:
        check_node = before
        for _ in range(k):
            check_node = check_node.next
            if not check_node:
                return dummy.next
        start = before.next
        for _ in range(k - 1):
            move = start.next
            start.next = move.next
            move.next = before.next
            before.next = move
        before = start

assert to_list(answer_5_reverse_k_group(build_list([1, 2, 3, 4, 5]), 2)) == [2, 1, 4, 3, 5]
assert to_list(answer_5_reverse_k_group(build_list([1, 2, 3, 4, 5]), 3)) == [3, 2, 1, 4, 5]
assert to_list(answer_5_reverse_k_group(build_list([1, 2, 3]), 1)) == [1, 2, 3]
for _ in range(2000):
    n, k = random.randint(1, 12), random.randint(1, 5)
    v = list(range(n))
    expected = []
    for i in range(0, n, k):
        chunk = v[i:i + k]
        expected += chunk[::-1] if len(chunk) == k else chunk
    assert to_list(answer_5_reverse_k_group(build_list(v), k)) == expected
print("Bài 5 PASS")

Bài 5 PASS


# 15. Tổng kết

> Dummy → đi `left − 1` bước tới `before` → `start = before.next` → lặp `right − left` lần: lấy `start.next`, chèn ngay sau `before`. Một lượt, `O(1)` bộ nhớ.

---

## Checklist trước khi nộp

- [ ] Có nút giả (để `left = 1` không phải trường hợp riêng)?
- [ ] Đi đúng `left − 1` bước?
- [ ] Chèn lên đầu: lặp `right − left` lần (cách cắt-nối: `right − left + 1`)?
- [ ] Cách cắt-nối: có `start.next = cur`?